# boxed: QLoRA post-training Qwen2.5-1.5B on NuminaMath

Post-trains [`Qwen/Qwen2.5-1.5B`](https://huggingface.co/Qwen/Qwen2.5-1.5B) (base, Apache-2.0) with 4-bit QLoRA on a **decontaminated** slice of [NuminaMath-CoT](https://huggingface.co/datasets/project-numina/NuminaMath-CoT), then measures the GSM8K delta against the untrained baseline.

**Why "decontaminated" matters.** NuminaMath-CoT contains 7,345 rows sourced from GSM8K itself. Training on those and then evaluating on GSM8K measures recall of the benchmark, not reasoning. `src/prepare_data.py` drops every `source == "gsm8k"` row before sampling, and the baseline eval runs *before* training so the comparison is honest.

Repo: https://github.com/Raufnarejo505/boxed

**Before running anything**, set in the right-hand panel: **Accelerator = GPU T4 x2**, **Internet = On**, **Persistence = Files only**. Total runtime is roughly 2 hours.

Run the cells top to bottom. Do not skip the baseline eval — without it there is no result, only a fine-tuned model.

## 1. Environment check

Confirm you actually got a GPU and have disk space before spending quota.

In [ ]:
import shutil, subprocess, torch

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
    capture_output=True, text=True).stdout)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print(f"free disk (/kaggle/working): {shutil.disk_usage('/kaggle/working').free / 1e9:.1f} GB")

assert torch.cuda.is_available(), "No GPU. Set Accelerator = GPU T4 x2 in the sidebar."

## 2. Configure

Edit these two values, then run every cell below.

In [ ]:
REPO_URL = "https://github.com/Raufnarejo505/boxed.git"
HF_USER  = "abdulrauf5121"   # must match the Hugging Face account that owns HF_TOKEN

## 3. Get the code and dependencies

`requirements.txt` pins `trl==0.12.2` deliberately: later versions renamed the
`tokenizer` argument to `processing_class` and the trainer call breaks without it.

In [ ]:
%cd /kaggle/working
!git clone $REPO_URL boxed || echo "repo already cloned"
%cd /kaggle/working/boxed
!pip install -q -r requirements.txt
print("installed")

**Restart the kernel now** (Run -> Restart) if `pip` upgraded `transformers` or `torch`.
Kaggle preloads its own versions into memory, and a stale `transformers` produces
confusing import errors. After restarting, run the next cell to get back into the repo
directory, then continue.

In [ ]:
%cd /kaggle/working/boxed

## 4. Confirm the Hub repositories

`configs/boxed-1.5b.yaml` already points at `abdulrauf5121/boxed-1.5b-lora` and
`abdulrauf5121/boxed-1.5b-merged`. This cell rewrites those to match `HF_USER`, so a
fork or a different account only needs the variable in step 2 changed.

In [ ]:
from pathlib import Path

cfg = Path("configs/boxed-1.5b.yaml")
text = cfg.read_text()
text = text.replace("abdulrauf5121/boxed-1.5b-lora", f"{HF_USER}/boxed-1.5b-lora")
text = text.replace("abdulrauf5121/boxed-1.5b-merged", f"{HF_USER}/boxed-1.5b-merged")
cfg.write_text(text)

for name in ("model_card.md", "README.md"):
    p = Path(name)
    p.write_text(p.read_text().replace("abdulrauf5121", HF_USER))

print(cfg.read_text().split("hub:")[-1])

## 5. Hugging Face login

Requires a **Write** token stored as the Kaggle secret `HF_TOKEN`
(Add-ons -> Secrets). Never paste a token into a cell.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

login(token=UserSecretsClient().get_secret("HF_TOKEN"))
print("logged in as:", whoami()["name"])

## 6. Verify before spending GPU minutes

These two suites take seconds. `test_api_compat.py` checks our code against the
*installed* trl/peft/transformers, which is how a `tokenizer` vs `processing_class`
drift gets caught in 5 seconds instead of 40 minutes into training.

In [ ]:
!python tests/test_smoke.py
!python tests/test_api_compat.py

## 7. Build the training file

Drops the 7,345 GSM8K-derived rows, filters by length, samples 20,000. Takes a few
minutes and is CPU-bound.

In [ ]:
!python src/prepare_data.py --out data/sft_train.jsonl --n 20000

## 8. BASELINE eval — run this before training

This is the number the entire project hangs on. Roughly 10 minutes. Write it down.

In [ ]:
!python src/eval_gsm8k.py --model Qwen/Qwen2.5-1.5B --limit 250 --out results/base.json

## 9. Train

~1-1.5 hours on one T4. Checkpoints every 200 steps into `outputs/boxed-1.5b`.
If you hit CUDA OOM, set `per_device_train_batch_size: 4` and
`gradient_accumulation_steps: 4` in the config and restart from this cell.

In [ ]:
!python src/train_sft.py --config configs/boxed-1.5b.yaml

## 10. Evaluate the adapter

Same model, same 250 questions, now with the LoRA weights attached.

In [ ]:
!python src/eval_gsm8k.py --model Qwen/Qwen2.5-1.5B --adapter outputs/boxed-1.5b --limit 250 --out results/lora.json

## 11. The result

Copy these two numbers into `README.md` and `model_card.md`.

In [ ]:
import json

for name in ("base", "lora"):
    d = json.load(open(f"results/{name}.json"))
    print(f"{name:5s}  {d['accuracy']:.4f}   ({d['n_correct']}/{d['n']})")

base = json.load(open("results/base.json"))["accuracy"]
lora = json.load(open("results/lora.json"))["accuracy"]
print(f"\ndelta: {100 * (lora - base):+.1f} points")

## 12. Publish

Pushes the adapter first (~20 MB, so work is never lost), then merges into fp32 and
pushes fp16 weights (~3 GB). Requires the repo names in the config to be yours.

In [ ]:
!python src/push_to_hub.py --config configs/boxed-1.5b.yaml --adapter outputs/boxed-1.5b

## 13. Persist the notebook

`/kaggle/working` is wiped when the session ends. Before closing:

1. **Save Version -> Save & Run All (Commit)** — keeps the outputs and `results/*.json`.
2. Confirm both repos exist at `huggingface.co/<your-user>/boxed-1.5b-lora` and `-merged`.
3. Commit `results/base.json` and `results/lora.json` back to GitHub — the eval logs are
   the evidence behind the README table.

Then make this notebook public once the results table is filled in.